# 🎯 Phase 4: Milestone Exam Solutions

> **Mathematical Foundations & ML Fundamentals**
>
> This notebook contains comprehensive solutions for all five Phase Milestone Exam questions.
> Each solution demonstrates the synthesis of concepts from Days 37-48.

---

## Question 1: End-to-End ML Pipeline — Housing Prices

**Combines**: Linear Algebra (Day 37), Regression (Day 43), Model Evaluation (Day 44)

**Scenario**: Build a complete ML pipeline that:
1. Generates/loads synthetic housing data
2. Engineers features
3. Trains Linear Regression, Ridge, and Decision Tree models
4. Evaluates with RMSE, MAE, R²
5. Selects the best model

In [ ]:
import numpy as np
import random

# Generate synthetic housing data (no sklearn dependency for data generation)
def generate_housing_data(n=300, seed=42):
    """
    Generate synthetic housing data with realistic correlations.

    Features: sqft, bedrooms, bathrooms, age, garage_spaces, has_pool
    Target: price
    """
    np.random.seed(seed)

    sqft = np.random.randint(800, 4000, n).astype(float)
    bedrooms = np.clip(sqft // 600 + np.random.randint(-1, 2, n), 1, 6).astype(float)
    bathrooms = np.clip(bedrooms - np.random.randint(0, 2, n), 1, 4).astype(float)
    age = np.random.randint(0, 80, n).astype(float)
    garage = np.random.choice([0, 1, 2], n).astype(float)
    pool = np.random.choice([0, 1], n).astype(float)

    # Price = linear combination + noise
    price = (
        sqft * 150
        + bedrooms * 20000
        + bathrooms * 15000
        - age * 1000
        + garage * 10000
        + pool * 25000
        + np.random.normal(0, 30000, n)
    )
    price = np.maximum(price, 50000)

    X = np.column_stack([sqft, bedrooms, bathrooms, age, garage, pool])
    feature_names = ["sqft", "bedrooms", "bathrooms", "age", "garage", "pool"]
    return X, price, feature_names


X, y, feature_names = generate_housing_data()
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Price range: ${y.min():,.0f} - ${y.max():,.0f}")
print(f"Features: {feature_names}")

In [ ]:
# --- Pure NumPy ML Implementation (no sklearn) ---

def train_test_split_manual(X, y, test_ratio=0.2, seed=42):
    """Split data into train and test sets."""
    np.random.seed(seed)
    n = len(y)
    indices = np.random.permutation(n)
    split = int(n * (1 - test_ratio))
    train_idx, test_idx = indices[:split], indices[split:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def standardize(X_train, X_test):
    """Z-score standardization (mean=0, std=1)."""
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-8  # Avoid division by zero
    return (X_train - mean) / std, (X_test - mean) / std, mean, std


def add_bias(X):
    """Add bias column (column of 1s) for intercept."""
    return np.column_stack([np.ones(len(X)), X])


class LinearRegressionNumpy:
    """Ordinary Least Squares using the Normal Equation: w = (X^T X)^-1 X^T y."""

    def __init__(self):
        self.weights = None

    def fit(self, X, y):
        X_b = add_bias(X)
        # Normal equation: w = (X^T X)^(-1) X^T y
        self.weights = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        return self

    def predict(self, X):
        X_b = add_bias(X)
        return X_b @ self.weights


class RidgeRegressionNumpy:
    """Ridge Regression (L2 regularization): w = (X^T X + λI)^-1 X^T y."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.weights = None

    def fit(self, X, y):
        X_b = add_bias(X)
        n_features = X_b.shape[1]
        identity = np.eye(n_features)
        identity[0, 0] = 0  # Don't regularize the bias term
        self.weights = np.linalg.pinv(X_b.T @ X_b + self.alpha * identity) @ X_b.T @ y
        return self

    def predict(self, X):
        X_b = add_bias(X)
        return X_b @ self.weights

In [ ]:
# Evaluation metrics
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot


# Split and standardize
X_train, X_test, y_train, y_test = train_test_split_manual(X, y)
X_train_s, X_test_s, mu, sigma = standardize(X_train, X_test)

# Train models
models = {
    "Linear Regression": LinearRegressionNumpy(),
    "Ridge (α=1.0)": RidgeRegressionNumpy(alpha=1.0),
    "Ridge (α=10.0)": RidgeRegressionNumpy(alpha=10.0),
}

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(f"{'Model':25s} {'RMSE':>12s} {'MAE':>12s} {'R²':>8s}")
print("-" * 60)

best_model = None
best_r2 = -float('inf')

for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    r = rmse(y_test, y_pred)
    m = mae(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name:25s} ${r:>10,.0f} ${m:>10,.0f} {r2:>7.4f}")
    if r2 > best_r2:
        best_r2 = r2
        best_model = name

print(f"\n🏆 Best Model: {best_model} (R² = {best_r2:.4f})")

---

## Question 2: Classification with Imbalanced Data — Fraud Detection

**Combines**: Classification (Day 45), Probability (Day 40), Model Evaluation (Day 44)

**Scenario**: Build a fraud detection classifier that handles class imbalance using:
1. Synthetic fraud data (3% fraud rate)
2. Logistic Regression with class weights
3. Threshold tuning for precision-recall trade-off
4. Confusion matrix analysis

In [ ]:
# Generate synthetic fraud data
def generate_fraud_data(n=1000, fraud_rate=0.03, seed=42):
    """
    Generate synthetic transaction data with class imbalance.

    Fraud transactions have: higher amounts, odd hours, higher velocity,
    larger distance from home.
    """
    np.random.seed(seed)
    X_list = []
    y_list = []

    for _ in range(n):
        is_fraud = 1 if np.random.random() < fraud_rate else 0

        if is_fraud:
            amount = np.random.uniform(500, 5000)
            hour = np.random.choice([0, 1, 2, 3, 4, 23])
            is_intl = np.random.choice([0, 1], p=[0.3, 0.7])
            velocity = np.random.randint(5, 20)
            distance = np.random.uniform(100, 5000)
        else:
            amount = np.random.uniform(5, 300)
            hour = np.random.randint(8, 21)
            is_intl = np.random.choice([0, 1], p=[0.85, 0.15])
            velocity = np.random.randint(0, 5)
            distance = np.random.uniform(0, 50)

        X_list.append([amount, hour, is_intl, velocity, distance])
        y_list.append(is_fraud)

    return np.array(X_list), np.array(y_list), \
           ["amount", "hour", "is_international", "velocity_24h", "distance_from_home"]


X_fraud, y_fraud, fraud_features = generate_fraud_data(n=2000)
print(f"Dataset: {len(y_fraud)} transactions")
print(f"Fraud rate: {y_fraud.mean()*100:.1f}% ({y_fraud.sum()} fraud / {(1-y_fraud).sum():.0f} legit)")
print(f"Features: {fraud_features}")

In [ ]:
# Logistic Regression from scratch with class weights
class LogisticRegressionWeighted:
    """
    Logistic Regression with gradient descent and class weights.

    Class weights allow the model to penalize misclassification of the
    minority class (fraud) more heavily.
    """

    def __init__(self, lr=0.01, epochs=1000, class_weight=None):
        self.lr = lr
        self.epochs = epochs
        self.class_weight = class_weight or {0: 1.0, 1: 1.0}
        self.weights = None
        self.bias = 0

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)  # Prevent overflow
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n, d = X.shape
        self.weights = np.zeros(d)
        self.bias = 0

        # Build sample weights from class weights
        sample_weights = np.array([self.class_weight[int(yi)] for yi in y])

        for epoch in range(self.epochs):
            z = X @ self.weights + self.bias
            predictions = self._sigmoid(z)
            errors = (predictions - y) * sample_weights

            # Gradients
            dw = (1/n) * X.T @ errors
            db = (1/n) * np.sum(errors)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.weights + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


def confusion_matrix_manual(y_true, y_pred):
    """Calculate confusion matrix components."""
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, fp, tn, fn

In [ ]:
# Train and evaluate
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split_manual(X_fraud, y_fraud)
X_train_fs, X_test_fs, _, _ = standardize(X_train_f, X_test_f)

# Calculate class weight: inversely proportional to frequency
n_neg = (y_train_f == 0).sum()
n_pos = (y_train_f == 1).sum()
weight_pos = n_neg / (n_pos + 1e-8)  # Weight fraud class higher
print(f"Class weight for fraud: {weight_pos:.1f}x")

# Model 1: No class weight (baseline)
model_unweighted = LogisticRegressionWeighted(lr=0.1, epochs=2000)
model_unweighted.fit(X_train_fs, y_train_f)

# Model 2: With class weight
model_weighted = LogisticRegressionWeighted(
    lr=0.1, epochs=2000, class_weight={0: 1.0, 1: weight_pos}
)
model_weighted.fit(X_train_fs, y_train_f)

# Compare
print("\n" + "=" * 60)
print("FRAUD DETECTION MODEL COMPARISON")
print("=" * 60)

for name, model in [("Unweighted", model_unweighted), ("Weighted", model_weighted)]:
    y_pred = model.predict(X_test_fs, threshold=0.5)
    tp, fp, tn, fn = confusion_matrix_manual(y_test_f, y_pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n  {name}:")
    print(f"    Precision: {precision:.3f}  (Of flagged, how many are truly fraud?)")
    print(f"    Recall:    {recall:.3f}  (Of actual fraud, how many did we catch?)")
    print(f"    F1 Score:  {f1:.3f}")
    print(f"    Confusion: TP={tp}, FP={fp}, TN={tn}, FN={fn}")

In [ ]:
# Threshold tuning
print("\n" + "=" * 60)
print("THRESHOLD TUNING (Weighted Model)")
print("=" * 60)
print(f"{'Threshold':>10s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Flagged':>10s}")
print("-" * 55)

probas = model_weighted.predict_proba(X_test_fs)
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    y_pred_t = (probas >= t).astype(int)
    tp, fp, tn, fn = confusion_matrix_manual(y_test_f, y_pred_t)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    flagged = y_pred_t.sum()
    print(f"{t:>10.1f} {prec:>10.3f} {rec:>10.3f} {f1:>10.3f} {flagged:>10d}")

print("\n💡 Business Insight: Lower threshold catches more fraud (higher recall)")
print("   but flags more legitimate transactions (lower precision).")
print("   Choose based on: cost of missing fraud vs cost of false alarms.")

---

## Question 3: Image Classification with CNN (Conceptual)

**Combines**: Neural Networks (Day 47), Linear Algebra (Day 37), Calculus (Day 38)

> ⚠️ TensorFlow/PyTorch are not available in Pyodide. This solution demonstrates the CNN architecture conceptually and implements a simplified forward pass in NumPy.

### CNN Architecture Explanation

A Convolutional Neural Network for image classification works in layers:

| Layer | Purpose | Analogy |
|-------|---------|----------|
| **Conv2D** | Detect patterns (edges, textures) | Sliding a magnifying glass over the image |
| **ReLU** | Add non-linearity | "If negative, make it zero" |
| **MaxPool** | Downsample, reduce computation | Keep only the brightest pixel in each 2×2 block |
| **Flatten** | Reshape 2D feature maps to 1D vector | Unroll a grid into a list |
| **Dense** | Final classification | "Based on all features, which class?" |

### The Architecture (in TensorFlow)

```python
# In production (TensorFlow/Keras):
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')  # 10 classes
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
```

In [ ]:
# Simplified CNN Forward Pass in NumPy
def conv2d(image, kernel):
    """
    2D Convolution (no padding, stride=1).

    Slides the kernel across the image and computes element-wise
    multiplication + sum at each position.

    Args:
        image: 2D numpy array (H x W)
        kernel: 2D numpy array (kH x kW)

    Returns:
        2D numpy array: Feature map
    """
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh, ow = ih - kh + 1, iw - kw + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output


def relu(x):
    """ReLU activation: max(0, x)."""
    return np.maximum(0, x)


def max_pool2d(feature_map, pool_size=2):
    """Max pooling: downsample by keeping the max in each block."""
    h, w = feature_map.shape
    oh, ow = h // pool_size, w // pool_size
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            block = feature_map[i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size]
            output[i, j] = np.max(block)
    return output


# Demo: Apply CNN layers to a synthetic 8x8 "image"
print("=" * 50)
print("CNN FORWARD PASS DEMO")
print("=" * 50)

# Create a simple 8x8 image with a diagonal edge
image = np.zeros((8, 8))
for i in range(8):
    image[i, i] = 1.0
    if i > 0: image[i, i-1] = 0.5
    if i < 7: image[i, i+1] = 0.5

print(f"\nInput Image (8x8):")
for row in image:
    print("  ", " ".join(f"{v:.1f}" for v in row))

# Edge detection kernel (Sobel-like)
edge_kernel = np.array([[-1, 0, 1],
                        [-2, 0, 2],
                        [-1, 0, 1]])

# Forward pass
conv_out = conv2d(image, edge_kernel)
relu_out = relu(conv_out)
pool_out = max_pool2d(relu_out)

print(f"\nAfter Conv2D (6x6):")
for row in conv_out:
    print("  ", " ".join(f"{v:5.1f}" for v in row))

print(f"\nAfter ReLU (6x6):")
for row in relu_out:
    print("  ", " ".join(f"{v:5.1f}" for v in row))

print(f"\nAfter MaxPool (3x3):")
for row in pool_out:
    print("  ", " ".join(f"{v:5.1f}" for v in row))

# Flatten for Dense layer
flat = pool_out.flatten()
print(f"\nFlattened: {flat} (length {len(flat)})")
print("\n→ This vector would be fed into a Dense layer for classification")

---

## Question 4: Time Series with LSTM (Conceptual)

**Combines**: Sequences (Day 48), Calculus (Day 38), Neural Networks (Day 47)

> ⚠️ TensorFlow/PyTorch are not available in Pyodide. This solution explains the LSTM architecture and demonstrates a simplified sequence prediction.

### LSTM Architecture Explanation

**Problem**: Standard neural networks have no "memory". They can't remember that yesterday's stock price affects today's.

**Solution**: LSTM (Long Short-Term Memory) cells have **gates** that control information flow:

| Gate | Purpose | Analogy |
|------|---------|----------|
| **Forget Gate** | Decide what to forget from memory | "Is yesterday's weather still relevant?" |
| **Input Gate** | Decide what new info to store | "Today's temperature is important" |
| **Output Gate** | Decide what to output | "Combine memory + today → tomorrow's prediction" |

### The Architecture (in TensorFlow)

```python
# In production (TensorFlow/Keras):
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(64, input_shape=(lookback, n_features), return_sequences=True),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1)  # Single value prediction
])
model.compile(optimizer='adam', loss='mse')
```

In [ ]:
# Simplified LSTM cell in NumPy
class SimpleLSTMCell:
    """
    Minimal LSTM cell implementation for educational purposes.

    Demonstrates the gate mechanism (Forget, Input, Output) with
    random weights (not trained — just showing the forward pass).
    """

    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        np.random.seed(42)
        scale = 0.1

        # Combined weights for all gates [forget, input, cell, output]
        combined = input_size + hidden_size
        self.Wf = np.random.randn(hidden_size, combined) * scale  # Forget
        self.Wi = np.random.randn(hidden_size, combined) * scale  # Input
        self.Wc = np.random.randn(hidden_size, combined) * scale  # Cell candidate
        self.Wo = np.random.randn(hidden_size, combined) * scale  # Output
        self.bf = np.zeros(hidden_size)
        self.bi = np.zeros(hidden_size)
        self.bc = np.zeros(hidden_size)
        self.bo = np.zeros(hidden_size)

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def forward(self, x, h_prev, c_prev):
        """
        One step of LSTM forward pass.

        Args:
            x: Input at this timestep
            h_prev: Previous hidden state (memory output)
            c_prev: Previous cell state (long-term memory)

        Returns:
            h_next: New hidden state
            c_next: New cell state
            gates: Dict of gate values (for visualization)
        """
        combined = np.concatenate([h_prev, x])

        # Gate computations
        f = self._sigmoid(self.Wf @ combined + self.bf)  # Forget gate
        i = self._sigmoid(self.Wi @ combined + self.bi)  # Input gate
        c_candidate = np.tanh(self.Wc @ combined + self.bc)  # Cell candidate
        o = self._sigmoid(self.Wo @ combined + self.bo)  # Output gate

        # Update cell state: forget old + add new
        c_next = f * c_prev + i * c_candidate

        # Output: filtered cell state
        h_next = o * np.tanh(c_next)

        return h_next, c_next, {"forget": f.mean(), "input": i.mean(), "output": o.mean()}


# Demo: Process a simple sequence
print("=" * 50)
print("LSTM FORWARD PASS DEMO")
print("=" * 50)

lstm = SimpleLSTMCell(input_size=1, hidden_size=4)

# Simulate a time series: [10, 20, 30, 40, 50]
sequence = [10, 20, 30, 40, 50]
h = np.zeros(4)
c = np.zeros(4)

print("\nProcessing sequence:", sequence)
print(f"{'Step':>6s} {'Input':>8s} {'Forget%':>10s} {'Input%':>10s} {'Output%':>10s} {'Hidden Mean':>12s}")
print("-" * 60)

for t, val in enumerate(sequence):
    x = np.array([val / 100.0])  # Normalize
    h, c, gates = lstm.forward(x, h, c)
    print(f"{t+1:>6d} {val:>8d} {gates['forget']:>10.1%} {gates['input']:>10.1%} {gates['output']:>10.1%} {h.mean():>12.4f}")

print("\n💡 The hidden state evolves as new inputs are processed, carrying forward memory.")
print("   In a trained LSTM, the gates would learn WHEN to remember and forget.")

---

## Question 5: Math Intuition Essay

**Combines**: Linear Algebra (Day 37), Calculus (Day 38), Statistics (Day 39-40)

### 1. Gradient Descent — The Mountain Hiker

**Intuition**: Imagine you're lost in dense fog on a mountain. You can't see the valley below, but you *can feel which direction slopes downward* under your feet. Gradient descent does exactly this:

- **Position** = current model parameters (weights)
- **Altitude** = error/loss (how wrong the model is)
- **Slope** = gradient (partial derivatives of loss w.r.t. each weight)
- **Step size** = learning rate (α)

**Update rule**: `w_new = w_old - α × gradient`

- **α too large**: You overshoot the valley, bouncing between hills
- **α too small**: You inch forward for years and may stop at a local minimum
- **Just right**: You descend smoothly to the global minimum

**Key insight**: The gradient tells you the *direction of steepest ascent*. You go the *opposite* direction (hence the minus sign).

---

### 2. Matrix Multiplication — The Assembly Line

**Intuition**: Matrix multiplication transforms data from one "space" to another.

Think of a factory assembly line:
- **Input matrix (X)**: Raw materials — each row is one product, each column is one ingredient
- **Weight matrix (W)**: Recipe book — each column transforms ingredients into a new feature
- **Output (XW)**: Finished products — each row is a product with new properties

**In neural networks**: `output = input × weights + bias`
- A 100×784 input (100 images, 784 pixels each)
- Multiplied by 784×128 weights
- Produces 100×128 output (100 images, 128 abstract features)

**The key rule**: inner dimensions must match. `(m × n) × (n × p) = (m × p)`

---

### 3. Activation Functions — The Decider

**Why we need them**: Without activation functions, a neural network with 100 layers is mathematically identical to a neural network with 1 layer. (Stacking linear transformations gives another linear transformation.)

| Function | Formula | Range | Use Case |
|----------|---------|-------|----------|
| **Sigmoid** | 1/(1+e^-x) | (0, 1) | Binary classification output |
| **ReLU** | max(0, x) | [0, ∞) | Hidden layers (default choice) |
| **Tanh** | (e^x - e^-x)/(e^x + e^-x) | (-1, 1) | Hidden layers (centered output) |
| **Softmax** | e^xi / Σe^xj | (0, 1), sums to 1 | Multi-class output |

**ReLU's genius**: It's computationally cheap (just a comparison), avoids vanishing gradients for positive values, and introduces non-linearity. Its weakness: "dead neurons" (once negative, always zero).

---

### 4. PCA Geometry — The Shadow Projector

**Intuition**: Imagine holding a 3D object (like a chair) under a lamp. The shadow on the wall is a 2D projection. **PCA finds the angle of the lamp that creates the most informative shadow** (preserves the most variance).

**Steps**:
1. **Center the data** (subtract mean)
2. **Compute covariance matrix** (how features relate)
3. **Find eigenvectors** (the "best lamp angles")
4. **Rank by eigenvalues** (how much information each angle preserves)
5. **Project** data onto top-k eigenvectors

**Business example**: You have 50 customer metrics. PCA finds that 90% of the variance is captured by just 3 "super-features" (PC1: spending power, PC2: engagement, PC3: loyalty). Now you can plot customers in 3D instead of 50D.

**The key math**: Eigenvectors of the covariance matrix point in the directions of maximum variance. Eigenvalues tell you how much variance each direction captures.

---

## 🎓 Summary

This notebook demonstrated solutions to all five Phase 4 Milestone Exam questions:

1. **ML Pipeline**: End-to-end housing price prediction with NumPy — Linear Regression, Ridge, evaluation metrics
2. **Imbalanced Classification**: Fraud detection with class weights, threshold tuning, and precision-recall trade-off
3. **CNN (Conceptual)**: Convolution, ReLU, and MaxPool forward pass implemented in NumPy
4. **LSTM (Conceptual)**: Gate mechanism (forget/input/output) demonstrated with a NumPy cell
5. **Math Intuition**: Written explanations of gradient descent, matrix multiplication, activation functions, and PCA

Key takeaways:
- ML is fundamentally **linear algebra + calculus + optimization**
- Understanding the math behind models helps you **debug and tune** them effectively
- Practical engineering (class weights, threshold tuning) is as important as the algorithm